Read Bronze Delta


In [0]:
bronze_path='/Volumes/workspace/default/railway_bronze/data'

#already stored data 
bronze_df=(
    spark.read\
    .format('delta')\
    .load(bronze_path)
)

# verfiy records and column
print('bronze records:',bronze_df.count())
print('bronze columns:',len(bronze_df.columns))


display(bronze_df.limit(5))

#inspect the schema
bronze_df.printSchema()

Data Quality Profiling


In [0]:
from pyspark.sql import functions as F

# Count total records
total_records = bronze_df.count()

# to get  NULL counts for every column
null_counts_df = bronze_df.select([
    F.count(
        F.when(F.col(column_name).isNull(), 1)
    ).alias(column_name)
    for column_name in bronze_df.columns
])

# Convert the one-row result into a readable report
null_report = [
    (column_name, null_counts_df.first()[column_name])
    for column_name in bronze_df.columns
]

# Create a DataFrame containing the quality report
null_report_df = spark.createDataFrame(
    null_report,
    ["column_name", "null_count"]
)

# Display columns with the most NULL values first
display(
    null_report_df.orderBy(F.desc("null_count"))
)

print("Total Bronze Records:", total_records)

display the actual null counts

In [0]:
# Display NULL counts in descending order
display(
    null_report_df.orderBy(F.desc("null_count"))
)

Check mandatory fields


In [0]:
from pyspark.sql import functions as F

# Important columns for identifying railway movements
mandatory_columns = [
    "date",
    "train_no",
    "station_code",
    "seq"
]

# Count NULL values in mandatory columns
mandatory_null_df = bronze_df.select([
    F.sum(
        F.when(F.col(c).isNull(), 1).otherwise(0)
    ).alias(c)
    for c in mandatory_columns
])

# Display results
display(mandatory_null_df)

# Count records missing at least one mandatory field
invalid_key_df = bronze_df.filter(
    F.col("date").isNull() |
    F.col("train_no").isNull() |
    F.col("station_code").isNull() |
    F.col("seq").isNull()
)

print("Records with missing mandatory fields:", invalid_key_df.count())

Check duplicate railway records

In [0]:
from pyspark.sql import functions as F

# Columns used to identify a railway movement
key_columns = [
    "date",
    "train_no",
    "station_code",
    "seq"
]

# Group records by their identifying columns
duplicate_df = (
    bronze_df
    .groupBy(*key_columns)
    .count()

    # Keep only keys appearing more than once
    .filter(F.col("count") > 1)

    # Show most frequently repeated keys first
    .orderBy(F.desc("count"))
)

# Number of duplicate key groups
print("Duplicate Key Groups:", duplicate_df.count())

# Number of records beyond one per duplicate key
extra_duplicate_records = duplicate_df.select(
    F.sum(F.col("count") - 1).alias("extra_records")
).first()["extra_records"]

print("Extra Duplicate Records:", extra_duplicate_records or 0)

# Display duplicate key combinations
display(duplicate_df.limit(20))

Investigate duplicate records


In [0]:

from pyspark.sql import functions as F

# Key columns used in the previous duplicate check
key_columns = [
    "date",
    "train_no",
    "station_code",
    "seq"
]

# Count exact duplicates across all original railway columns
# Exclude ingestion metadata because timestamps can change on reruns
data_columns = [
    c for c in bronze_df.columns
    if c not in [
        "source_file",
        "ingestion_timestamp",
        "pipeline_name"
    ]
]

# Count completely identical railway records
exact_duplicates = (
    bronze_df.count()
    - bronze_df.dropDuplicates(data_columns).count()
)

print("Exact Duplicate Records:", exact_duplicates)

# Identify repeated keys where railway data differs
conflicting_keys_df = (
    bronze_df
    .groupBy(*key_columns)
    .agg(
        F.count("*").alias("total_records"),
        F.countDistinct(
            F.struct(*[F.col(c) for c in data_columns])
        ).alias("distinct_versions")
    )
    .filter(
        (F.col("total_records") > 1) &
        (F.col("distinct_versions") > 1)
    )
)

print("Repeated Keys With Different Data:",
      conflicting_keys_df.count())

# Display sample conflicting records
display(conflicting_keys_df.limit(10))

Investigate conflicting observations

In [0]:
from pyspark.sql import functions as F

# Join conflicting keys back to the original Bronze data
conflicting_records_df = (
    bronze_df
    .join(
        conflicting_keys_df.select(
            "date",
            "train_no",
            "station_code",
            "seq"
        ),
        on=["date", "train_no", "station_code", "seq"],
        how="inner"
    )
)

# Count records with and without reporting timestamps
conflicting_records_df.select(
    F.count("*").alias("total_conflicting_records"),

    F.sum(
        F.when(F.col("reported_at").isNull(), 1)
         .otherwise(0)
    ).alias("missing_reported_at"),

    F.sum(
        F.when(F.col("reported_at").isNotNull(), 1)
         .otherwise(0)
    ).alias("available_reported_at")
).show()

# Inspect a sample of conflicting observations
display(
    conflicting_records_df.select(
        "date",
        "train_no",
        "station_code",
        "seq",
        "act_arrival",
        "act_departure",
        "delay_arrival_min",
        "delay_departure_min",
        "reported_at"
    ).orderBy(
        "date",
        "train_no",
        "station_code",
        "seq",
        F.col("reported_at").desc_nulls_last()
    ).limit(20)
)

Check timestamp reliability

In [0]:
from pyspark.sql import functions as F

# Group conflicting records by railway movement key
timestamp_quality_df = (
    conflicting_records_df
    .groupBy(
        "date",
        "train_no",
        "station_code",
        "seq"
    )
    .agg(
        # Total observations for each movement
        F.count("*").alias("total_records"),

        # Observations containing a reporting timestamp
        F.count("reported_at").alias("timestamp_available"),

        # Number of different reporting timestamps
        F.countDistinct("reported_at").alias("distinct_timestamps")
    )
)

# Summarize timestamp reliability across conflicting groups
timestamp_quality_df.select(

    # Total conflicting key groups
    F.count("*").alias("total_conflicting_groups"),

    # Groups where every observation has a timestamp
    F.sum(
        F.when(
            F.col("timestamp_available") == F.col("total_records"),
            1
        ).otherwise(0)
    ).alias("fully_timestamped_groups"),

    # Groups containing missing timestamps
    F.sum(
        F.when(
            F.col("timestamp_available") < F.col("total_records"),
            1
        ).otherwise(0)
    ).alias("groups_with_missing_timestamps"),

    # Groups with at least two distinct timestamps
    F.sum(
        F.when(
            F.col("distinct_timestamps") > 1,
            1
        ).otherwise(0)
    ).alias("groups_with_multiple_timestamps")

).show(truncate=False)

Check invalid numerical values

In [0]:


from pyspark.sql import functions as F

# Check numerical data quality
numeric_quality_df = bronze_df.select(

    # Negative distance is physically invalid
    F.sum(
        F.when(F.col("distance_km") < 0, 1).otherwise(0)
    ).alias("negative_distance"),

    # Station sequence should be positive
    F.sum(
        F.when(F.col("seq") <= 0, 1).otherwise(0)
    ).alias("invalid_sequence"),

    # Very large arrival delays need investigation
    F.sum(
        F.when(F.abs(F.col("delay_arrival_min")) > 1440, 1)
         .otherwise(0)
    ).alias("extreme_arrival_delay"),

    # Very large departure delays need investigation
    F.sum(
        F.when(F.abs(F.col("delay_departure_min")) > 1440, 1)
         .otherwise(0)
    ).alias("extreme_departure_delay")
)

# Display numerical quality results
display(numeric_quality_df)

Investigate sequence values

In [0]:
from pyspark.sql import functions as F

# Count NULL, zero, negative and positive sequences
sequence_quality_df = bronze_df.select(

    # Missing sequence numbers
    F.count(
        F.when(F.col("seq").isNull(), 1)
    ).alias("null_sequence"),

    # Sequence equal to zero
    F.count(
        F.when(F.col("seq") == 0, 1)
    ).alias("zero_sequence"),

    # Negative sequence numbers
    F.count(
        F.when(F.col("seq") < 0, 1)
    ).alias("negative_sequence"),

    # Valid positive sequence numbers
    F.count(
        F.when(F.col("seq") > 0, 1)
    ).alias("positive_sequence")
)

# Display results
display(sequence_quality_df)

Investigate zero sequence records

In [0]:

from pyspark.sql import functions as F

# Select records where station sequence is zero
zero_seq_df = bronze_df.filter(
    F.col("seq") == 0
)

# Count zero-sequence records
print("Zero Sequence Records:", zero_seq_df.count())

# Check how often zero sequence appears at each station
display(
    zero_seq_df
    .groupBy("station_code")
    .count()
    .orderBy(F.desc("count"))
    .limit(15)
)

# Inspect sample records for railway context
display(
    zero_seq_df.select(
        "date",
        "train_no",
        "train_name",
        "station_code",
        "seq",
        "sch_departure",
        "distance_km"
    ).limit(10)
)

Check whether zero represents the starting station

In [0]:


from pyspark.sql import functions as F

# Filter records where sequence is zero
zero_seq_df = bronze_df.filter(F.col("seq") == 0)

# Analyze distance values for zero-sequence records
zero_seq_df.select(

    # Total zero-sequence records
    F.count("*").alias("total_records"),

    # Records with distance equal to zero
    F.count(
        F.when(F.col("distance_km") == 0, 1)
    ).alias("zero_distance"),

    # Records with positive distance
    F.count(
        F.when(F.col("distance_km") > 0, 1)
    ).alias("positive_distance"),

    # Records with missing distance
    F.count(
        F.when(F.col("distance_km").isNull(), 1)
    ).alias("missing_distance")

).show()

Create a sequence quality classification

In [0]:


from pyspark.sql import functions as F

# Add a quality status for the station sequence
quality_df = (
    bronze_df
    .withColumn(
        "sequence_quality",
        F.when(
            F.col("seq").isNull(),
            F.lit("MISSING_SEQUENCE")
        )
        .when(
            F.col("seq") <= 0,
            F.lit("INVALID_OR_UNKNOWN_SEQUENCE")
        )
        .otherwise(
            F.lit("VALID_SEQUENCE")
        )
    )
)

# Summarize records by sequence quality
display(
    quality_df
    .groupBy("sequence_quality")
    .count()
    .orderBy(F.desc("count"))
)

Classify duplicate and conflicting records

In [0]:


from pyspark.sql import functions as F

# Railway movement identification columns
key_columns = [
    "date",
    "train_no",
    "station_code",
    "seq"
]

# Original railway columns, excluding ingestion metadata
data_columns = [
    c for c in bronze_df.columns
    if c not in [
        "source_file",
        "ingestion_timestamp",
        "pipeline_name"
    ]
]

# Count distinct railway observations per movement key
key_versions_df = (
    bronze_df
    .groupBy(*key_columns)
    .agg(
        F.count("*").alias("key_record_count"),
        F.countDistinct(
            F.struct(*[F.col(c) for c in data_columns])
        ).alias("distinct_versions")
    )
)

# Join classification back to the existing quality DataFrame
quality_df = (
    quality_df
    .join(
        key_versions_df,
        on=key_columns,
        how="left"
    )
    .withColumn(
        "duplicate_status",
        F.when(
            F.col("distinct_versions") > 1,
            F.lit("CONFLICTING_OBSERVATIONS")
        )
        .when(
            F.col("key_record_count") > 1,
            F.lit("EXACT_DUPLICATE_GROUP")
        )
        .otherwise(
            F.lit("UNIQUE_RECORD")
        )
    )
)

# Display classification summary
display(
    quality_df
    .groupBy("duplicate_status")
    .count()
    .orderBy(F.desc("count"))
)

Prepare valid and review datasets

In [0]:

from pyspark.sql import functions as F

# Assign each record to one processing category
classified_df = (
    quality_df
    .withColumn(
        "quality_status",

        # Conflicting observations require investigation
        F.when(
            F.col("duplicate_status") == "CONFLICTING_OBSERVATIONS",
            F.lit("REVIEW")
        )

        # Missing or nonpositive sequences require quarantine
        .when(
            F.col("sequence_quality") != "VALID_SEQUENCE",
            F.lit("QUARANTINE")
        )

        # Other records are candidates for Silver
        .otherwise(
            F.lit("SILVER_CANDIDATE")
        )
    )
)

# Summarize the classification
display(
    classified_df
    .groupBy("quality_status")
    .count()
    .orderBy("quality_status")
)

# Verify all Bronze records are accounted for
print("Classified Records:", classified_df.count())

 Display classification counts

In [0]:
# Display records in each quality category
classified_df.groupBy("quality_status") \
    .count() \
    .orderBy("quality_status") \
    .show(truncate=False)

Remove exact duplicates from Silver candidates

In [0]:
from pyspark.sql import functions as F

# Select Silver candidates from the quality-classified Bronze records
silver_candidate_df = classified_df.filter(
    F.col("quality_status") == "SILVER_CANDIDATE"
)

# Deduplicate using the ORIGINAL railway columns, excluding ingestion metadata
# (data_columns is defined in the earlier duplicate-classification cell)
before_count = silver_candidate_df.count()
silver_dedup_df = silver_candidate_df.dropDuplicates(data_columns)
after_count = silver_dedup_df.count()
duplicates_removed = before_count - after_count

print("Before Deduplication:", before_count)
print("After Deduplication:", after_count)
print("Duplicates Removed:", duplicates_removed)

Validate cleaned Silver candidates

In [0]:


from pyspark.sql import functions as F

# Step 1: Count cleaned Silver records
silver_count = silver_dedup_df.count()

# Step 2: Check mandatory identifiers
mandatory_nulls = silver_dedup_df.filter(
    F.col("date").isNull() |
    F.col("train_no").isNull() |
    F.col("station_code").isNull() |
    F.col("seq").isNull()
).count()

# Step 3: Check invalid sequence values
invalid_sequences = silver_dedup_df.filter(
    F.col("seq") <= 0
).count()

# Step 4: Check duplicate movement keys
duplicate_keys = (
    silver_dedup_df
    .groupBy("date", "train_no", "station_code", "seq")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

# Step 5: Check negative distances
negative_distances = silver_dedup_df.filter(
    F.col("distance_km") < 0
).count()

# Step 6: Display validation results
print("Silver Records:", silver_count)
print("Mandatory NULL Records:", mandatory_nulls)
print("Invalid Sequence Records:", invalid_sequences)
print("Duplicate Movement Key Groups:", duplicate_keys)
print("Negative Distance Records:", negative_distances)

 SAVE DATA QUALITY OUTPUTS

In [0]:
from pyspark.sql import functions as F

# Step 1: Define output paths in Silver Volume
silver_base_path = "/Volumes/workspace/default/railway_silver"

silver_clean_path = silver_base_path + "/clean"
quarantine_path = silver_base_path + "/quarantine"
review_path = silver_base_path + "/review"

# Step 2: Extract quarantine records
quarantine_df = classified_df.filter(
    F.col("quality_status") == "QUARANTINE"
)

# Step 3: Extract conflicting observations
review_df = classified_df.filter(
    F.col("quality_status") == "REVIEW"
)

# Step 4: Save cleaned Silver candidates
silver_dedup_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(silver_clean_path)

# Step 5: Save quarantined records
quarantine_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(quarantine_path)

# Step 6: Save conflicting records
review_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(review_path)

# Step 7: Verify persisted record counts
for name, path in [
    ("Silver Clean", silver_clean_path),
    ("Quarantine", quarantine_path),
    ("Review", review_path)
]:
    count = spark.read.format("delta").load(path).count()
    print(f"{name}: {count} records")

Independent duplicate audit (after Silver writes)


In [0]:
from pyspark.sql import functions as F

# ------------------------------------------
# 1. DEFINE STORAGE PATHS
# ------------------------------------------

bronze_path = "/Volumes/workspace/default/railway_bronze/data"
clean_path = "/Volumes/workspace/default/railway_silver/clean"
quarantine_path = "/Volumes/workspace/default/railway_silver/quarantine"
review_path = "/Volumes/workspace/default/railway_silver/review"

audit_path = "/Volumes/workspace/default/railway_silver/duplicate_audit"


# ------------------------------------------
# 2. READ PERSISTED DATASETS
# ------------------------------------------

bronze_check = spark.read.format("delta").load(bronze_path)
clean_check = spark.read.format("delta").load(clean_path)
quarantine_check = spark.read.format("delta").load(quarantine_path)
review_check = spark.read.format("delta").load(review_path)


# ------------------------------------------
# 3. COUNT RECORDS
# ------------------------------------------

bronze_total = bronze_check.count()
clean_total = clean_check.count()
quarantine_total = quarantine_check.count()
review_total = review_check.count()


# ------------------------------------------
# 4. CALCULATE ACTUAL DUPLICATES
# ------------------------------------------

# Reuse the original deduplication DataFrames
# created earlier in 02_Data_Quality.

candidate_total = silver_candidate_df.count()
deduplicated_total = silver_dedup_df.count()

actual_duplicates_removed = (
    candidate_total - deduplicated_total
)


# ------------------------------------------
# 5. VERIFY RECONCILIATION
# ------------------------------------------

unaccounted_records = (
    bronze_total
    - clean_total
    - quarantine_total
    - review_total
)

assert actual_duplicates_removed >= 0, (
    "Invalid duplicate count"
)

assert actual_duplicates_removed == unaccounted_records, (
    f"Duplicate reconciliation failed: "
    f"Actual={actual_duplicates_removed}, "
    f"Unaccounted={unaccounted_records}"
)


# ------------------------------------------
# 6. CREATE AUDIT RECORD
# ------------------------------------------

audit_records = [(
    bronze_total,
    clean_total,
    quarantine_total,
    review_total,
    actual_duplicates_removed
)]

audit_columns = [
    "bronze_records",
    "silver_clean_records",
    "quarantine_records",
    "review_records",
    "duplicates_removed"
]

audit_df = spark.createDataFrame(
    audit_records,
    audit_columns
).withColumn(
    "audit_timestamp",
    F.current_timestamp()
)


# ------------------------------------------
# 7. SAVE DUPLICATE AUDIT
# ------------------------------------------

audit_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(audit_path)


# ------------------------------------------
# 8. DISPLAY RESULTS
# ------------------------------------------

display(audit_df)

print("DUPLICATE AUDIT: PASSED")
print("Before Deduplication:", candidate_total)
print("After Deduplication:", deduplicated_total)
print("Actual Duplicates Removed:", actual_duplicates_removed)
print("Unaccounted Records:", unaccounted_records)
print("Audit Path:", audit_path)

Final Data Quality reconciliation

In [0]:
# Step 1: Read persisted Delta datasets
bronze_count = spark.read.format("delta") \
    .load("/Volumes/workspace/default/railway_bronze/data") \
    .count()

silver_count = spark.read.format("delta") \
    .load("/Volumes/workspace/default/railway_silver/clean") \
    .count()

quarantine_count = spark.read.format("delta") \
    .load("/Volumes/workspace/default/railway_silver/quarantine") \
    .count()

review_count = spark.read.format("delta") \
    .load("/Volumes/workspace/default/railway_silver/review") \
    .count()

# Step 2: Count duplicates removed from Silver candidates
duplicates_removed = (
    silver_candidate_df.count() - silver_count
)

# Step 3: Reconcile all records
reconciled_count = (
    silver_count
    + quarantine_count
    + review_count
    + duplicates_removed
)

# Step 4: Display reconciliation
print("Bronze Records:", bronze_count)
print("Silver Clean:", silver_count)
print("Quarantine:", quarantine_count)
print("Review:", review_count)
print("Duplicates Removed:", duplicates_removed)
print("Reconciled Records:", reconciled_count)

# Step 5: Validate reconciliation
assert bronze_count == reconciled_count, \
    "FAILED: Record count mismatch!"

print("DATA QUALITY RECONCILIATION: PASSED")